# Data Generation for SFT, DPO, RFT

In [ ]:
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

# TODO: fill in your resource group name and OpenAI API key
RESOURCE_GROUP = "cis-5270-team-10"
OPENAI_API_KEY = ""

OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = ""

os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"]  = "CIS-5270"
os.environ["AZURE_AOAI_ACCOUNT"]    = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT

CREDENTIAL = DefaultAzureCredential()

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)

# TODO: replace with actual deployment name once TA provisions it
TEACHER_DEPLOYMENT = "gpt-4.1-mini"

print("Connected to Azure OpenAI")

Connected to Azure OpenAI


## SFT Data Generation
Use GPT-4.1-mini to write a justification for each (passage, claim, label) in the training pool.
The label comes from FEVER ground truth, mini just needs to explain *why*.

In [55]:
import json
import time
from collections import Counter

In [ ]:
# load the training pool
train_pool = []
with open("data/joined/fever_train_joined.jsonl") as f:
    for line in f:
        train_pool.append(json.loads(line))

print(f"loaded {len(train_pool)} examples")
from collections import Counter
print(Counter(ex["label"] for ex in train_pool))

In [12]:
SYSTEM_PROMPT = """You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a one-sentence justification quoting or closely paraphrasing the passage.
Format: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"""

def get_justification(passage, claim, label):
    # label is still passed so mini knows the ground-truth answer to justify
    user_msg = f"Passage: {passage}\n\nClaim: {claim}\n\nVerdict: {label}"
    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=TEACHER_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0.3,
                max_tokens=150,
            )
            text = resp.choices[0].message.content.strip()
            # strip label prefix if mini echoes it back (e.g. "SUPPORTED: ...")
            for lbl in ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]:
                if text.upper().startswith(lbl + ":"):
                    text = text[len(lbl)+1:].strip()
                    break
            return text
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            import time; time.sleep(2 ** attempt)
    return None

In [13]:
# quick connection test, call mini on one example before running the full loop
test_ex = train_pool[0]
test_result = get_justification(test_ex["passage"], test_ex["claim"], test_ex["label"])
print("passage:      ", test_ex["passage"][:100])
print("claim:        ", test_ex["claim"])
print("label:        ", test_ex["label"])
print("justification:", test_result)

passage:       Underworld is a 2003 action horror film directed by Len Wiseman and written by Danny McBride , based
claim:         Underworld is a religion.
label:         CONTRADICTED
justification: The passage states that Underworld is a "2003 action horror film," which contradicts the claim that it is a religion.


In [ ]:
# generate justifications and save in inference-compatible format
# assistant target = "LABEL: justification", matches predict_sft extraction at eval time
SFT_OUT = "data/generated/sft_data.jsonl"

done_ids = set()
if os.path.exists(SFT_OUT):
    with open(SFT_OUT) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"resuming, {len(done_ids)} already done")

skipped = 0
with open(SFT_OUT, "a") as out_f:
    for i, ex in enumerate(train_pool):
        if ex["id"] in done_ids:
            continue

        justification = get_justification(ex["passage"], ex["claim"], ex["label"])
        if justification is None:
            skipped += 1
            continue

        record = {
            "id": ex["id"],
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": f"Passage: {ex['passage']}\n\nClaim: {ex['claim']}"},
                {"role": "assistant", "content": f"{ex['label']}: {justification}"},
            ],
            "label": ex["label"],
            "justification": justification,
            "passage": ex["passage"],
            "claim": ex["claim"],
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()

        if (i + 1) % 500 == 0:
            print(f"[{i+1}/{len(train_pool)}] skipped={skipped}")

print(f"done. skipped {skipped}")

In [ ]:
# quick sanity check
sft_data = []
with open(SFT_OUT) as f:
    for line in f:
        sft_data.append(json.loads(line))

print(f"total SFT examples: {len(sft_data)}")
print(Counter(ex["label"] for ex in sft_data))
print("\nsample:")
print(json.dumps(sft_data[0]["messages"], indent=2))

---
## DPO Data Generation

Each DPO example is a `(prompt, chosen, rejected)` triple.
- **chosen**: correct label + grounded justification (reused from `sft_data.jsonl`)
- **rejected type 1**: wrong label, zero-shot mini, keep only where it mislabels
- **rejected type 2**: correct label + hallucinated justification, explicitly prompt mini to fabricate

We generate 2500 chosen, 1250 type-1 negatives, 1250 type-2 negatives (per spec).

In [ ]:
import random
random.seed(42)

# load sft_data as the source of positives (chosen)
sft_data = []
with open("data/generated/sft_data.jsonl") as f:
    for line in f:
        sft_data.append(json.loads(line))

# shuffle and split: 2500 for chosen/type-1, 1250 for type-2
random.shuffle(sft_data)
pool_neg1 = sft_data[:1250]
pool_neg2 = sft_data[1250:2500]
pool_chosen = sft_data[:2500]

print(f"chosen pool: {len(pool_chosen)}")
print(f"neg type-1 pool: {len(pool_neg1)}")
print(f"neg type-2 pool: {len(pool_neg2)}")

In [ ]:
# Negative Type 1: wrong label, swapped justification
# no API call needed, pick a random wrong label and steal the justification from a different example
# the justification is real/grounded but for the wrong claim, making it an unfaithful negative

OTHER_LABELS = {
    "SUPPORTED":     ["CONTRADICTED", "NOT MENTIONED"],
    "CONTRADICTED":  ["SUPPORTED",    "NOT MENTIONED"],
    "NOT MENTIONED": ["SUPPORTED",    "CONTRADICTED"],
}

# index sft_data by label for easy sampling
from collections import defaultdict
sft_by_label = defaultdict(list)
for ex in sft_data:
    sft_by_label[ex["label"]].append(ex)

def get_wrong_label_negative(ex):
    wrong_label = random.choice(OTHER_LABELS[ex["label"]])
    # borrow a justification written for the wrong label from a different example
    donor = random.choice(sft_by_label[wrong_label])
    return wrong_label, donor["justification"]

In [ ]:
NEG1_OUT = "data/generated/dpo_neg1.jsonl"

with open(NEG1_OUT, "w") as out_f:
    for ex in pool_neg1:
        wrong_label, wrong_just = get_wrong_label_negative(ex)
        record = {
            "id":                     ex["id"],
            "passage":                ex["passage"],
            "claim":                  ex["claim"],
            "ground_truth":           ex["label"],
            "rejected_label":         wrong_label,
            "rejected_justification": wrong_just,
        }
        out_f.write(json.dumps(record) + "\n")

print(f"done. {len(pool_neg1)} type-1 negatives written to {NEG1_OUT}")

In [58]:
# Negative Type 2: unsupported inference justification
# correct label, but justification makes an inference that is not fully supported by the passage
# introduces subtle reasoning errors such as overgeneralization, overextension, or causal overreach
# no external facts should be invented, only incorrect reasoning over the given text

HALLUC_SYSTEM = """You are generating REJECTED justifications for preference optimization.

Write exactly ONE sentence.

The justification must be plausible but subtly wrong.

Rules by verdict:

If verdict = SUPPORTED:
- The justification MUST still argue that the claim is supported.
- Do NOT reject, dispute, or undermine the claim.
- The flaw must come only from invented support or overclaimed reasoning.

If verdict = CONTRADICTED:
- Write a justification that incorrectly tries to rescue the claim using conflation, weak association, or faulty reasoning.

If verdict = NOT MENTIONED:
- Write a justification that incorrectly infers the claim from irrelevant evidence.

Global rules:
- The error should be subtle, not absurd.
- Use exactly one reasoning flaw.
- Output only the justification sentence."""

def get_hallucinated_justification(passage, claim, label):
    flaw = random.choice(["OVERREACH","FABRICATED_DETAIL","CONFLATION"])
    user_msg = f"Passage: {passage}\n\nClaim: {claim}\n\nVerdict: {label}\n\nFlaw type: {flaw}"
    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=TEACHER_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": HALLUC_SYSTEM},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0.7,
                max_tokens=150,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return None

In [51]:
# spot-check both negative prompts before running full loops
test_cases = random.sample(sft_data, 3)

print("=== NEG2: hallucinated justification ===")
for ex in test_cases:
    result = get_hallucinated_justification(ex["passage"], ex["claim"], ex["label"])
    print(f"  label: {ex['label']}")
    print(f"  claim:   {ex['claim']}")
    print(f"  passage: {ex['passage'][:80]}")
    print(f"  result:  {result}")
    print()

=== NEG2: hallucinated justification ===
  label: NOT MENTIONED
  claim:   Get Out has grossed $241 million every week.
  passage: Fürstenberg-Geisingen was a county in southern Baden-Württemberg , Germany durin
  result:  The passage’s mention of a specific historical region in Germany subtly suggests a longstanding cultural influence that could be linked to modern entertainment successes like the weekly gross of $241 million by Get Out.

  label: SUPPORTED
  claim:   Naomi Watts is a thespian.
  passage: Naomi Ellen Watts -LRB- born 28 September 1968 -RRB- is an English actress and f
  result:  Since Naomi Watts is described as an English actress and film producer, it is clear that she is a thespian because all actresses automatically qualify as thespians without exception.

  label: SUPPORTED
  claim:   The Hateful Eight is a film by Quentin Tarantino.
  passage: The Hateful Eight is a 2015 American revisionist Western mystery film written an
  result:  The claim is supported becaus

In [ ]:
NEG2_OUT = "data/generated/dpo_neg2.jsonl"

done_ids = set()
if os.path.exists(NEG2_OUT):
    with open(NEG2_OUT) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"resuming, {len(done_ids)} already done")

skipped = 0
with open(NEG2_OUT, "a") as out_f:
    for i, ex in enumerate(pool_neg2):
        if ex["id"] in done_ids:
            continue

        halluc = get_hallucinated_justification(ex["passage"], ex["claim"], ex["label"])
        if halluc is None:
            skipped += 1
            continue

        record = {
            "id":                     ex["id"],
            "passage":                ex["passage"],
            "claim":                  ex["claim"],
            "ground_truth":           ex["label"],
            "rejected_label":         ex["label"],
            "rejected_justification": halluc,
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()

        if (i + 1) % 200 == 0:
            print(f"[{i+1}/{len(pool_neg2)}] skipped={skipped}")

print(f"done. type-2 negatives: {len(pool_neg2), skipped}  (skipped {skipped})")

In [ ]:
# Assemble final DPO pairs
DPO_OUT = "data/generated/dpo_data.jsonl"

chosen_by_id = {ex["id"]: ex for ex in pool_chosen}

neg1 = []
with open(NEG1_OUT) as f:
    for line in f:
        neg1.append(json.loads(line))

neg2 = []
with open(NEG2_OUT) as f:
    for line in f:
        neg2.append(json.loads(line))

print(f"type-1 negatives: {len(neg1)}")
print(f"type-2 negatives: {len(neg2)}")

dpo_pairs = []
for neg in neg1 + neg2:
    ex_id = neg["id"]
    if ex_id not in chosen_by_id:
        continue
    chosen_ex = chosen_by_id[ex_id]

    prompt = f"Passage: {neg['passage']}\n\nClaim: {neg['claim']}"
    chosen_response   = f"{chosen_ex['label']}: {chosen_ex['justification']}"
    rejected_response = f"{neg['rejected_label']}: {neg['rejected_justification']}"

    dpo_pairs.append({
        "id":       ex_id,
        "prompt":   prompt,
        "chosen":   chosen_response,
        "rejected": rejected_response,
    })

random.shuffle(dpo_pairs)
with open(DPO_OUT, "w") as f:
    for pair in dpo_pairs:
        f.write(json.dumps(pair) + "\n")

print(f"\ntotal DPO pairs: {len(dpo_pairs)} → saved to {DPO_OUT}")

---
## DPO Data (3k pairs) from sft_data_3k

3000 pairs: 1500 neg1 (wrong label + stolen justification, no API calls) + 1500 neg2 (correct label + hallucinated justification, ~1143 new API calls).
Reuses any neg2 already generated in `dpo_neg2.jsonl`.

In [56]:
import random
from collections import defaultdict
random.seed(42)

# load sft_data_3k as the source (run Downsample.ipynb first)
sft_3k = []
with open("data/generated/sft_data_3k.jsonl") as f:
    for line in f:
        sft_3k.append(json.loads(line))

print(f"sft_data_3k loaded: {len(sft_3k)}")

# shuffle and split 1500/1500 for neg1 and neg2
random.shuffle(sft_3k)
pool_neg1_3k = sft_3k[:1500]
pool_neg2_3k = sft_3k[1500:3000]
pool_chosen_3k = sft_3k  # all 3000 are chosen

print(f"neg1 pool: {len(pool_neg1_3k)}")
print(f"neg2 pool: {len(pool_neg2_3k)}")
print(f"chosen pool: {len(pool_chosen_3k)}")

sft_data_3k loaded: 3000
neg1 pool: 1500
neg2 pool: 1500
chosen pool: 3000


In [57]:
# neg1: wrong label + stolen justification (no API calls)
NEG1_3K_OUT = "data/generated/dpo_neg1_3k.jsonl"

sft_3k_by_label = defaultdict(list)
for ex in sft_3k:
    sft_3k_by_label[ex["label"]].append(ex)

OTHER_LABELS = {
    "SUPPORTED":     ["CONTRADICTED", "NOT MENTIONED"],
    "CONTRADICTED":  ["SUPPORTED",    "NOT MENTIONED"],
    "NOT MENTIONED": ["SUPPORTED",    "CONTRADICTED"],
}

def get_wrong_label_negative_3k(ex):
    wrong_label = random.choice(OTHER_LABELS[ex["label"]])
    donor = random.choice(sft_3k_by_label[wrong_label])
    return wrong_label, donor["justification"]

with open(NEG1_3K_OUT, "w") as out_f:
    for ex in pool_neg1_3k:
        wrong_label, wrong_just = get_wrong_label_negative_3k(ex)
        record = {
            "id":                     ex["id"],
            "passage":                ex["passage"],
            "claim":                  ex["claim"],
            "ground_truth":           ex["label"],
            "rejected_label":         wrong_label,
            "rejected_justification": wrong_just,
        }
        out_f.write(json.dumps(record) + "\n")

print(f"done. {len(pool_neg1_3k)} type-1 negatives written to {NEG1_3K_OUT}")

done. 1500 type-1 negatives written to data/generated/dpo_neg1_3k.jsonl


In [59]:
# neg2: correct label + hallucinated justification
# reuses existing dpo_neg2.jsonl where IDs overlap, generates the rest via API
NEG2_3K_OUT = "data/generated/dpo_neg2_3k.jsonl"

pool_neg2_ids = {ex["id"] for ex in pool_neg2_3k}

# seed from existing neg2 where IDs match
existing_neg2 = {}
if os.path.exists("data/generated/dpo_neg2.jsonl"):
    with open("data/generated/dpo_neg2.jsonl") as f:
        for line in f:
            ex = json.loads(line)
            if ex["id"] in pool_neg2_ids:
                existing_neg2[ex["id"]] = ex

print(f"reusing {len(existing_neg2)} existing neg2 entries")

# also check if neg2_3k already has some done (resume support)
done_ids = set(existing_neg2.keys())
if os.path.exists(NEG2_3K_OUT):
    with open(NEG2_3K_OUT) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"resuming, {len(done_ids)} already done")

# write existing hits first
with open(NEG2_3K_OUT, "w") as out_f:
    for ex in existing_neg2.values():
        out_f.write(json.dumps(ex) + "\n")

# generate the rest
skipped = 0
with open(NEG2_3K_OUT, "a") as out_f:
    for i, ex in enumerate(pool_neg2_3k):
        if ex["id"] in done_ids:
            continue
        halluc = get_hallucinated_justification(ex["passage"], ex["claim"], ex["label"])
        if halluc is None:
            skipped += 1
            continue
        record = {
            "id":                     ex["id"],
            "passage":                ex["passage"],
            "claim":                  ex["claim"],
            "ground_truth":           ex["label"],
            "rejected_label":         ex["label"],
            "rejected_justification": halluc,
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()
        if (i + 1) % 200 == 0:
            print(f"[{i+1}/{len(pool_neg2_3k)}] skipped={skipped}")

print(f"done. skipped {skipped}. written to {NEG2_3K_OUT}")

reusing 357 existing neg2 entries
[200/1500] skipped=0
[800/1500] skipped=0
[1400/1500] skipped=0
done. skipped 0. written to data/generated/dpo_neg2_3k.jsonl


In [60]:
# assemble final 3k DPO pairs
DPO_3K_OUT = "data/generated/dpo_data_3k.jsonl"

chosen_by_id_3k = {ex["id"]: ex for ex in pool_chosen_3k}

neg1_3k = [json.loads(l) for l in open(NEG1_3K_OUT)]
neg2_3k = [json.loads(l) for l in open(NEG2_3K_OUT)]

print(f"type-1 negatives: {len(neg1_3k)}")
print(f"type-2 negatives: {len(neg2_3k)}")

dpo_pairs_3k = []
for neg in neg1_3k + neg2_3k:
    ex_id = neg["id"]
    if ex_id not in chosen_by_id_3k:
        continue
    chosen_ex = chosen_by_id_3k[ex_id]
    prompt = f"Passage: {neg['passage']}\n\nClaim: {neg['claim']}"
    chosen_response   = f"{chosen_ex['label']}: {chosen_ex['justification']}"
    rejected_response = f"{neg['rejected_label']}: {neg['rejected_justification']}"
    dpo_pairs_3k.append({
        "id":       ex_id,
        "prompt":   prompt,
        "chosen":   chosen_response,
        "rejected": rejected_response,
    })

random.shuffle(dpo_pairs_3k)
with open(DPO_3K_OUT, "w") as f:
    for pair in dpo_pairs_3k:
        f.write(json.dumps(pair) + "\n")

print(f"\ntotal DPO pairs: {len(dpo_pairs_3k)} saved to {DPO_3K_OUT}")

type-1 negatives: 1500
type-2 negatives: 1500

total DPO pairs: 3000 saved to data/generated/dpo_data_3k.jsonl


---
## Validation Set Generation

Small stratified sample from `fever_dev_joined.jsonl` (no overlap with train pool).
Teacher generates justifications in the same format as SFT training data.
Upload this as the validation file when submitting fine-tuning jobs.

In [61]:
import random
from collections import defaultdict, Counter
random.seed(99)  # different seed so sample is independent of train pool

VAL_SIZE = 300   # 100 per class
VAL_OUT  = "data/generated/sft_val.jsonl"

# load train pool IDs to exclude
train_pool_ids = set()
with open("data/generated/train_pool_3k.jsonl") as f:
    for line in f:
        train_pool_ids.add(json.loads(line)["id"])

# load dev, exclude any IDs in train pool (shouldn't overlap, but be safe)
dev_data = []
with open("data/joined/fever_dev_joined.jsonl") as f:
    for line in f:
        ex = json.loads(line)
        if ex["id"] not in train_pool_ids:
            dev_data.append(ex)

print(f"dev examples available (after train exclusion): {len(dev_data)}")

# stratified sample
buckets = defaultdict(list)
for ex in dev_data:
    buckets[ex["label"]].append(ex)

per_class = VAL_SIZE // len(buckets)
val_pool = []
for lbl, items in sorted(buckets.items()):
    random.shuffle(items)
    val_pool.extend(items[:per_class])
    print(f"  {lbl}: took {per_class}")

random.shuffle(val_pool)
print(f"\nval pool: {len(val_pool)} examples")
print(Counter(ex["label"] for ex in val_pool))

dev examples available (after train exclusion): 19891
  CONTRADICTED: took 100
  NOT MENTIONED: took 100
  SUPPORTED: took 100

val pool: 300 examples
Counter({'SUPPORTED': 100, 'NOT MENTIONED': 100, 'CONTRADICTED': 100})


In [62]:
# generate justifications for val pool (same format as sft_data)
done_ids = set()
if os.path.exists(VAL_OUT):
    with open(VAL_OUT) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"resuming, {len(done_ids)} already done")

skipped = 0
with open(VAL_OUT, "a") as out_f:
    for i, ex in enumerate(val_pool):
        if ex["id"] in done_ids:
            continue
        justification = get_justification(ex["passage"], ex["claim"], ex["label"])
        if justification is None:
            skipped += 1
            continue
        record = {
            "id": ex["id"],
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": f"Passage: {ex['passage']}\n\nClaim: {ex['claim']}"},
                {"role": "assistant", "content": f"{ex['label']}: {justification}"},
            ],
            "label": ex["label"],
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()

print(f"done. {VAL_SIZE - skipped} val examples saved to {VAL_OUT}  (skipped {skipped})")

done. 300 val examples saved to data/generated/sft_val.jsonl  (skipped 0)


In [63]:
import json
with open("data/generated/sft_val.jsonl") as f_in, open("data/generated/sft_val_upload.jsonl", "w") as f_out:
    for line in f_in:
        ex = json.loads(line)
        f_out.write(json.dumps({"messages": ex["messages"]}) + "\n")
print("done")


done
